# S6E10 | Final Blend: CV Stack × Public #1

One file for the second final pick: a 50/50 average, in percentile-rank space, of my CV-chosen meta-stack
(nested CV 0.962234, public 0.96184) and the public #1 file (public 0.96213). The weight is fixed in advance, not
tuned on the leaderboard.

**Why a blend and not either file alone.** goodpjw2008 measured (Twin Stacks Part 16, seven pre-registered
submissions) that a 50/50 rank blend of two files scores about **0.082 × s²** above the average of the two, where `s`
is the standard deviation of their percentile-rank difference. The public #1 sits far from the CV stacks
(`s` ≈ 0.02), so the blend should beat the CV stack on the private board unless the public #1 is privately worse
than it by more than about 2 × 0.082 × s². The CV stack stays the first final pick, so the pair covers both cases.

**Inputs:** the competition data; my meta-stack notebook (its `submission.csv` sits next to `submission_twin.csv`);
and one source of the public #1 file: goodpjw2008's dataset `s6e10-goodpjw2008-extra-members-oof` (latest version,
`public_096213_1023.csv`, a pinned copy) or nina2025's dataset `ps-s6e10-02` (latest version, `0.96213c.csv`).
CPU, a few seconds.

**Credits:** the public #1 file is mehrankazeminia's recipe on nina2025's rank file (published by several authors);
the blend rule and the noise rule are goodpjw2008's.

In [ ]:
import glob, os
import numpy as np
import pandas as pd
from scipy.stats import rankdata

INPUT = '/kaggle/input'
ids = pd.read_csv(next(p for p in glob.glob(f'{INPUT}/**/sample_submission.csv', recursive=True)
                       if 'playground-series-s6e10' in p))['id'].values


def load(path):
    d = pd.read_csv(path)
    col = [c for c in d.columns if c != 'id'][0]
    assert d['id'].is_unique and set(d['id']) == set(ids), path
    v = d.set_index('id').loc[ids, col].values.astype(float)
    assert np.isfinite(v).all(), path
    return v


def pct(v):
    return (rankdata(v) - 0.5) / len(v)


def first(pattern, hint=''):
    hits = sorted(p for p in glob.glob(f'{INPUT}/**/{pattern}', recursive=True) if hint in p)
    return hits[0] if hits else None


ours = [p for p in glob.glob(f'{INPUT}/**/submission.csv', recursive=True)
        if os.path.exists(os.path.join(os.path.dirname(p), 'submission_twin.csv'))]
assert ours, 'attach the meta-stack notebook (its submission.csv sits next to submission_twin.csv)'
top = (first('public_096213_1023.csv', 'goodpjw2008-extra-members') or first('0.96213c.csv')
       or first('0.96213b.csv') or first('0.96213a.csv'))
assert top, 'attach the latest version of goodpjw2008-extra-members-oof or nina2025/ps-s6e10-02'
R_ours, R_top = pct(load(ours[0])), pct(load(top))

s = (R_top - R_ours).std()
blend = pct(0.5 * R_ours + 0.5 * R_top)
pd.DataFrame({'id': ids, 'satisfaction': blend}).to_csv('submission.csv', index=False, float_format='%.12f')
print(f'CV stack:   {ours[0]}\npublic #1:  {top}')
print(f'rank correlation {np.corrcoef(R_ours, R_top)[0, 1]:.6f} | rank-difference std s = {s:.5f}')
print(f'expected gain of the blend over the average of the two: 0.082 x s^2 = {0.082 * s * s * 1e5:.1f}e-5')
print(f'public-board noise of the blend vs the CV stack (0.003 x rank-move std): '
      f'{0.003 * (blend - R_ours).std() * 1e5:.1f}e-5')
print('submission.csv written: 50% CV stack + 50% public #1, percentile ranks')